In [14]:
## csv 파일 변환
import pandas as pd

file_path = "incom_amount_yearly_notfilter.xlsx"

df = pd.read_excel(file_path)
df.to_csv("income_data.csv", index=False, encoding="utf-8-sig")

print("CSV 변환 완료")

CSV 변환 완료


In [57]:
import pandas as pd
import numpy as np
import re
import os

# -----------------------------
# 0. 안전한 CSV 읽기
# -----------------------------
def read_csv_safe(file_path):
    try:
        return pd.read_csv(file_path, header=None, encoding="utf-8-sig")
    except UnicodeDecodeError:
        try:
            return pd.read_csv(file_path, header=None, encoding="cp949")
        except UnicodeDecodeError:
            return pd.read_csv(file_path, header=None, encoding="euc-kr")


# -----------------------------
# 1. 전처리 함수
# -----------------------------
def preprocess_income_csv(file_path, year):

    # 🔥 여기서 읽는다 (return 금지)
    df = read_csv_safe(file_path)

    # -----------------------------
    # 1. 3줄 헤더 처리
    # -----------------------------
    h1 = df.iloc[0]
    h2 = df.iloc[1]
    h3 = df.iloc[2]

    new_cols = []

    for c1, c2, c3 in zip(h1, h2, h3):
        c1 = str(c1).strip()
        c2 = str(c2).strip()
        c3 = str(c3).strip()

        if "구분" in c3:
            new_cols.append(c3)
        else:
            new_cols.append(f"{c1}_{c2}")

    df.columns = new_cols

    data = df.iloc[3:].copy()
    # -----------------------------
    # 실제 데이터
    # -----------------------------
    data = df.iloc[3:].copy()

    # -----------------------------
    # 3. 컬럼 찾기
    # -----------------------------
    def find_col(keyword):
        for col in data.columns:
            if keyword in col:
                return col
        return None

    col_sido = find_col("구분2")
    col_sigungu = find_col("구분3")
    col_count = find_col("신고인원")
    col_income = find_col("종합소득금액")

    if not all([col_sido, col_sigungu, col_count, col_income]):
        raise ValueError("필수 컬럼 탐색 실패")

    data = data[[col_sido, col_sigungu, col_count, col_income]]
    data.columns = ["sido", "sigungu", "count", "income"]

    # -----------------------------
    # 4. 시도 보정
    # -----------------------------
    # 시도 보정
    data["sido"] = data["sido"].replace("", np.nan).ffill()

    # 시군구 보정
    data["sigungu"] = data["sigungu"].replace("", np.nan)

    # 만약 sigungu가 없으면 sido에서 가져오는 경우
    data["sigungu"] = np.where(
        data["sigungu"].isna(),
        data["sido"],
        data["sigungu"]
    )
    data = data[~data["sigungu"].isin(data["sido"])]
    # -----------------------------
    # 5. 문자열 정리
    # -----------------------------
    for col in ["sido", "sigungu"]:
        data[col] = data[col].astype(str).str.strip()

    data = data[data["sigungu"].notna()]
    data = data[data["sigungu"] != ""]
    data = data[data["sigungu"].str.lower() != "nan"]
    # -----------------------------
    # 🔥 시군구 구조 보정 (핵심)
    # -----------------------------
    # 시도 채우기
    data["sido"] = data["sido"].replace("", np.nan).ffill()

    # 시군구 정리
    data["sigungu"] = data["sigungu"].replace("", np.nan)

    # 👉 시군구가 없는 행 제거 (중요)
    data = data[data["sigungu"].notna()]

    # 👉 시군구가 "시도"인 행 제거
    data = data[data["sigungu"] != data["sido"]]

    # -----------------------------
    # 6. 소계 제거 (세종 제외)
    # -----------------------------
    data = data[
        ~((data["sigungu"].str.contains("소계", na=False)) & (data["sido"] != "세종"))
    ]

    # -----------------------------
    # 7. 숫자 처리
    # -----------------------------
    for col in ["count", "income"]:
        data[col] = pd.to_numeric(
            data[col].astype(str).str.replace(",", "", regex=False),
            errors="coerce"
        ).fillna(0)

    # -----------------------------
    # 8. 연도
    # -----------------------------
    target_years = [2024,2023,2022,2021,2020,2019,2018]

    if year not in target_years:
        return pd.DataFrame()

    data["year"] = year
    

    # -----------------------------
    # 9. 1인당 소득
    # -----------------------------
    data["income_per_person"] = np.where(
        data["count"] > 0,
        data["income"] / data["count"],
        0
    )

    # -----------------------------
    # 10. 이상치 체크
    # -----------------------------
    abnormal = data[(data["count"] == 0) & (data["income"] > 0)]

    if len(abnormal) > 0:
        print("[경고] 이상 데이터 존재")

    # -----------------------------
    # 11. 컬럼 정리
    # -----------------------------
    data = data[
        ["year", "sido", "sigungu", "count", "income", "income_per_person"]
    ]

    return data.reset_index(drop=True)



In [58]:
# -----------------------------
# F. 전처리 실행 (여러 연도)
# -----------------------------
file_path = "income_data.csv"

years = [2024, 2023, 2022, 2021, 2020, 2019, 2018]

all_data = []

for y in years:
    temp = preprocess_income_csv(file_path, year=y)
    if not temp.empty:
        all_data.append(temp)

df_all = pd.concat(all_data, ignore_index=True)

print(df_all.head())
print("전처리 완료")


# -----------------------------
# 5. CSV 저장
# -----------------------------
df_all.to_csv("income_sigungu_yearly.csv", index=False, encoding="utf-8-sig")
print("파일 저장 완료")


# -----------------------------
# 6. 🔥 여기다 넣는다 (pivot)
# -----------------------------
pivot_table = df_all.pivot_table(
    index=["sido", "sigungu"],
    columns="year",
    values="income_per_person",
    aggfunc="mean"
)

print(pivot_table.head())

   year sido sigungu   count    income  income_per_person
0  2024   서울     강남구  179449  22169913         123.544366
1  2024   서울     강동구  129427   4596287          35.512582
2  2024   서울     강북구   71159   1493286          20.985202
3  2024   서울     강서구  146603   4270842          29.132023
4  2024   서울     관악구  142776   3293963          23.070845
전처리 완료
파일 저장 완료
year               2018       2019       2020       2021       2022  \
sido sigungu                                                          
강원   강릉시      28.202528  28.202528  28.202528  28.202528  28.202528   
     고성군      23.604976  23.604976  23.604976  23.604976  23.604976   
     동해시      27.021921  27.021921  27.021921  27.021921  27.021921   
     삼척시      26.578720  26.578720  26.578720  26.578720  26.578720   
     속초시      28.871428  28.871428  28.871428  28.871428  28.871428   

year               2023       2024  
sido sigungu                        
강원   강릉시      28.202528  28.202528  
     고성군      23.604976  23